## Import ##

In [4]:
# LOCATION : https://github.com/purnasai/Dino_V2
import torch
from torchvision import models, transforms
import cv2
import os
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from PIL import Image
import pickle
from time import gmtime, strftime

os.environ["XFORMERS_DISABLED"] = "1" # Switch to enable xFormers
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [5]:
import torch
import torch.nn as nn

DINO_PATH_FINETUNED_DOWNLOADED="/media/osero/SamsungSSD/colab_saved_hand_large_not_final/teacher_checkpoint.pth"

def get_dino_finetuned_downloaded():
    # load the original DINOv2 model with the correct architecture and parameters. The positional embedding is too large.
    # load vits or vitg
    model=torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14')
    #model=torch.hub.load('facebookresearch/dinov2', 'dinov2_vitg14')
    # load finetuned weights
    pretrained = torch.load(DINO_PATH_FINETUNED_DOWNLOADED, map_location=torch.device('cpu'))
    # make correct state dict for loading
    new_state_dict = {}
    for key, value in pretrained['teacher'].items():
        if 'dino_head' in key:
            print('not used')
        else:
            new_key = key.replace('backbone.', '')
            new_state_dict[new_key] = value
    #change shape of pos_embed, shape depending on vits or vitg
    # pos_embed = nn.Parameter(torch.zeros(1, 257, 384))
    pos_embed = nn.Parameter(torch.zeros(1, 257, 1024))
    model.pos_embed = pos_embed
    # load state dict
    model.load_state_dict(new_state_dict, strict=True)
    return model

In [6]:
# # Load DINO ViT model from torchvision (for example, ViT small or base model trained with DINO)
# import torch.nn as nn

# class VideoClassifierLSTM(nn.Module):
#     def __init__(self, num_classes):
#         super(VideoClassifierLSTM, self).__init__()
#         self.dino_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
#         self.fc = nn.Linear(self.dino_model.embed_dim, num_classes)
#         self.dropout = nn.Dropout(0.1)

#     def forward(self, x):        
#         dino_feature = self.dino_model(x)
#         output = self.dropout(dino_feature)
#         output = self.fc(output)  # Take hidden state of the last LSTM layer
#         return output
    
# video_lassifier_model = VideoClassifierLSTM(num_classes=744)

# video_lassifier_model.load_state_dict(torch.load("/home/osero/Desktop/CMPE/dinov2/classsification/lstm/lstm_results/FINE_TUNED_FACE_B_MODEL2025-01-01_18-05-14_3.pth"))

model = get_dino_finetuned_downloaded()
model.to(device)
model.eval()

# Define transform to match the input size for the model
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

Using cache found in /home/osero/.cache/torch/hub/facebookresearch_dinov2_main
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:45: UserWarning: xFormers is disabled (SwiGLU)
  warnings.warn("xFormers is disabled (SwiGLU)")
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:29: UserWarning: xFormers is disabled (Attention)
  warnings.warn("xFormers is disabled (Attention)")
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:35: UserWarning: xFormers is disabled (Block)
  warnings.warn("xFormers is disable

not used
not used
not used
not used
not used
not used
not used
not used


## Functions ##

In [7]:
def extract_video_embedding(image_list):
    """Extracts and averages embeddings for a list of frames."""
    embeddings = []
    with torch.no_grad():
        input_tensor = torch.stack(image_list).to(device)
        features = model(input_tensor)
        features = features.cpu().numpy()
        embeddings = [features[i, :] for i in range(features.shape[0])]

    return embeddings

In [11]:
# Step 3: Function to Extract Embeddings for a List of Frames
def extract_video_embedding_single(image_list):
    """Extracts and averages embeddings for a list of frames."""
    embeddings = []
    with torch.no_grad():
        for image in image_list:
            # Convert frame to PIL image and apply transformations
            input_tensor = image.unsqueeze(0).to(device)  # Add batch dimension
            features = model(input_tensor)
            embeddings.append(features.squeeze().cpu().numpy())
    # Average the embeddings to get a single representation for the video
    video_embedding = np.mean(embeddings, axis=0)
    return video_embedding

## Process ##

In [12]:
try:
    # Step 4: Process All Videos in the Dataset
    # Set paths to your video dataset and labels
    video_folder = "/media/osero/SamsungSSD/CMPE_SSD/frame-hand_left-c256" 
    video_labels = []  # Populate this with the corresponding labels for each video
    video_embeddings = []
    process_count = 0

    # Step 4: Process All Videos in the Dataset
    # Set paths to your video dataset and labels
    video_folder = "/media/osero/SamsungSSD/CMPE_SSD/frame-hand_left-c256" 
    pickle_saved_file_path = '/media/osero/SamsungSSD/features_left_hand_large_trained_mixed_saved.pickle'
    video_labels = []  # Populate this with the corresponding labels for each video
    video_embeddings = []
    initial_label = 0
    process_count = 0

    if os.path.isfile(pickle_saved_file_path):
        pickle_file11 = open(pickle_saved_file_path, 'rb')
        video_embeddings,video_labels = pickle.load(pickle_file11)
        initial_label = video_labels[-1]
        print("initial_label is: ", initial_label)


    for label_folder in sorted(os.listdir(video_folder)):
        process_count += 1
        full_label_folder = os.path.join(video_folder, label_folder)
        label = int(label_folder)
        if label <= initial_label:
            continue
        print("process_count: ", process_count, ' , label: ', label)
        for sample_folder in sorted(os.listdir(full_label_folder)):
            full_sample_folder = os.path.join(full_label_folder, sample_folder)
            image_list = []
            for image_file in sorted(os.listdir(full_sample_folder)):
                full_image_file = os.path.join(full_sample_folder, image_file)
                image = Image.open(full_image_file)
                image = transform(image)
                image_list.append(image)
            video_embedding = extract_video_embedding_single(image_list)
            video_embeddings.append(video_embedding)
            video_labels.append(label)
        if(process_count % 10 == 0):
            with open(pickle_saved_file_path, 'wb') as handle:
                pickle.dump((video_embeddings, video_labels), handle, protocol=pickle.HIGHEST_PROTOCOL)

    with open('/media/osero/SamsungSSD/CMPE_SSD/features_left_hand_large_trained_mixed.pickle', 'wb') as handle:
        pickle.dump((video_embeddings, video_labels), handle, protocol=pickle.HIGHEST_PROTOCOL)
    abc = 4
except Exception as error:
    # handle the exception
    print("An exception occurred:", error)



initial_label is:  720
process_count:  721  , label:  721
process_count:  722  , label:  722
process_count:  723  , label:  723
process_count:  724  , label:  724
process_count:  725  , label:  725
process_count:  726  , label:  726
process_count:  727  , label:  727
process_count:  728  , label:  728
process_count:  729  , label:  729
process_count:  730  , label:  730
process_count:  731  , label:  731
process_count:  732  , label:  732
process_count:  733  , label:  733
process_count:  734  , label:  734
process_count:  735  , label:  735
process_count:  736  , label:  736
process_count:  737  , label:  737
process_count:  738  , label:  738
process_count:  739  , label:  739
process_count:  740  , label:  740
process_count:  741  , label:  741
process_count:  742  , label:  742
process_count:  743  , label:  743
process_count:  744  , label:  744


In [13]:
try:
    # Step 4: Process All Videos in the Dataset
    # Set paths to your video dataset and labels
    video_folder = "/media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256" 
    pickle_saved_file_path = '/media/osero/SamsungSSD/features_right_hand_large_trained_mixed_saved.pickle'
    video_labels = []  # Populate this with the corresponding labels for each video
    video_embeddings = []
    initial_label = 0
    process_count = 0

    if os.path.isfile(pickle_saved_file_path):
        pickle_file11 = open(pickle_saved_file_path, 'rb')
        video_embeddings,video_labels = pickle.load(pickle_file11)
        initial_label = video_labels[-1]
        print("initial_label is: ", initial_label)

    for label_folder in sorted(os.listdir(video_folder)):
        process_count += 1
        full_label_folder = os.path.join(video_folder, label_folder)
        label = int(label_folder)
        if label <= initial_label:
            continue
        print("process_count: ", process_count, ' , label: ', label)
        for sample_folder in sorted(os.listdir(full_label_folder)):
            full_sample_folder = os.path.join(full_label_folder, sample_folder)
            image_list = []
            for image_file in sorted(os.listdir(full_sample_folder)):
                full_image_file = os.path.join(full_sample_folder, image_file)
                image = Image.open(full_image_file)
                image = transform(image)
                image_list.append(image)
            video_embedding = extract_video_embedding(image_list)
            video_embeddings.append(video_embedding)
            video_labels.append(label)
        if(process_count % 10 == 0):
            with open(pickle_saved_file_path, 'wb') as handle:
                pickle.dump((video_embeddings, video_labels), handle, protocol=pickle.HIGHEST_PROTOCOL)

    with open('/media/osero/SamsungSSD/CMPE_SSD/features_right_hand_large_trained_mixed.pickle', 'wb') as handle:
        pickle.dump((video_embeddings, video_labels), handle, protocol=pickle.HIGHEST_PROTOCOL)
    abc = 4

except Exception as error:
    # handle the exception
    print("An exception occurred:", error) # An exception occurred: division by zero


initial_label is:  250
process_count:  251  , label:  251
process_count:  252  , label:  252
process_count:  253  , label:  253
process_count:  254  , label:  254
process_count:  255  , label:  255
process_count:  256  , label:  256
process_count:  257  , label:  257
process_count:  258  , label:  258
process_count:  259  , label:  259
process_count:  260  , label:  260
process_count:  261  , label:  261
process_count:  262  , label:  262
process_count:  263  , label:  263
process_count:  264  , label:  264
process_count:  265  , label:  265
process_count:  266  , label:  266
process_count:  267  , label:  267
process_count:  268  , label:  268
process_count:  269  , label:  269
process_count:  270  , label:  270
process_count:  271  , label:  271
process_count:  272  , label:  272
process_count:  273  , label:  273
process_count:  274  , label:  274
process_count:  275  , label:  275
process_count:  276  , label:  276
process_count:  277  , label:  277
process_count:  278  , label:  2

In [14]:
# @@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
# @@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@ DEMO @@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
# @@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

try:
    # Step 4: Process All Videos in the Dataset
    # Set paths to your video dataset and labels
    video_folder = "/media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256" 
    pickle_saved_file_path = '/media/osero/SamsungSSD/features_right_hand_large_trained_mixed_saved.pickle'
    video_labels = []  # Populate this with the corresponding labels for each video
    video_embeddings = []
    initial_label = 0
    process_count = 0

    if os.path.isfile(pickle_saved_file_path):
        pickle_file11 = open(pickle_saved_file_path, 'rb')
        video_embeddings,video_labels = pickle.load(pickle_file11)
        initial_label = video_labels[-1]
        print("initial_label is: ", initial_label)

    for label_folder in sorted(os.listdir(video_folder)):
        process_count += 1
        full_label_folder = os.path.join(video_folder, label_folder)
        label = int(label_folder)
        if label <= initial_label:
            continue
        print("process_count: ", process_count, ' , label: ', label)
        for sample_folder in sorted(os.listdir(full_label_folder)):
            full_sample_folder = os.path.join(full_label_folder, sample_folder)
            image_list = []
            for image_file in sorted(os.listdir(full_sample_folder)):
                full_image_file = os.path.join(full_sample_folder, image_file)
                image = Image.open(full_image_file)
                image = transform(image)
                image_list.append(image)
            video_embedding = extract_video_embedding_single(image_list)
            video_embeddings.append(video_embedding)
            video_labels.append(label)
        if(process_count % 10 == 0):
            with open(pickle_saved_file_path, 'wb') as handle:
                pickle.dump((video_embeddings, video_labels), handle, protocol=pickle.HIGHEST_PROTOCOL)

    with open('/media/osero/SamsungSSD/CMPE_SSD/features_right_hand_large_trained_mixed.pickle', 'wb') as handle:
        pickle.dump((video_embeddings, video_labels), handle, protocol=pickle.HIGHEST_PROTOCOL)
    abc = 4

except Exception as error:
    # handle the exception
    print("An exception occurred:", error) # An exception occurred: division by zero


initial_label is:  720
process_count:  721  , label:  721
process_count:  722  , label:  722
process_count:  723  , label:  723
process_count:  724  , label:  724
process_count:  725  , label:  725
process_count:  726  , label:  726
process_count:  727  , label:  727
process_count:  728  , label:  728
process_count:  729  , label:  729
process_count:  730  , label:  730
process_count:  731  , label:  731
process_count:  732  , label:  732
process_count:  733  , label:  733
process_count:  734  , label:  734
process_count:  735  , label:  735
process_count:  736  , label:  736
process_count:  737  , label:  737
process_count:  738  , label:  738
process_count:  739  , label:  739
process_count:  740  , label:  740
process_count:  741  , label:  741
process_count:  742  , label:  742
process_count:  743  , label:  743
process_count:  744  , label:  744


## Evaluation ##

In [7]:
# with open('features_face_frames_small.pickle', 'wb') as handle:
#     pickle.dump((video_embeddings, video_labels), handle, protocol=pickle.HIGHEST_PROTOCOL)

In [3]:
pickle_file11 = open('features_right_hand_large_trained_mixed_saved.pickle', 'rb')
features11,labels11 = pickle.load(pickle_file11)
tcc = 5

KeyboardInterrupt: 